In [ ]:
# =============================================================================
# CELL 1 — Cài đặt & import
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gspread

import os
import re
import inspect
import shutil
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import joblib
from torch.utils.data import Dataset as TorchDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, cohen_kappa_score, mean_absolute_error, confusion_matrix
)
from scipy.stats import chisquare, binomtest, pearsonr, chi2_contingency
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, TrainerCallback
)

pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang dùng device: {DEVICE}")

import transformers, torchvision
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

BASE_DIR = "/content/drive/MyDrive/SCHOOL/KLTN"
CKPT_DIR = "/content/tmp_run"          # ổ tạm Colab, xoá sau mỗi fold
MODEL_SAVE_DIR = f"{BASE_DIR}/saved_models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# --- Cấu hình lưu model, chọn 1 trong 3 ---
SAVE_MODE = "best_per_seed"
SAVE_AS_FP16 = True

CLASS_ORDER = ["non-depressed", "mild", "moderate", "severe"]
RANDOM_SEED = 42

CHECKPOINT = "distilbert-base-uncased"
MAX_LEN = 128
LEARNING_RATE = 3e-5
BATCH_SIZE = 16
WARMUP_RATIO = 0.20
WEIGHT_DECAY = 0.01
N_SEEDS_TIER2 = 3
SEEDS = [1, 42, 2024][:N_SEEDS_TIER2]
N_FOLDS = 5
MAX_EPOCHS = 10
EARLY_STOP_PATIENCE = 2

label2id = {cls: i for i, cls in enumerate(CLASS_ORDER)}
id2label = {i: cls for cls, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)


def save_trained_model(trainer_or_model, name, as_fp16=SAVE_AS_FP16):
    model = trainer_or_model.model if hasattr(trainer_or_model, "model") else trainer_or_model
    state_dict = model.state_dict()
    if as_fp16:
        state_dict = {k: (v.half() if torch.is_tensor(v) and v.is_floating_point() else v)
                      for k, v in state_dict.items()}
    save_path = f"{MODEL_SAVE_DIR}/{name}_state_dict.pt"
    torch.save(state_dict, save_path)
    size_mb = os.path.getsize(save_path) / (1024 ** 2)
    print(f"  [lưu model] {name} -> {save_path} ({size_mb:.1f} MB, fp16={as_fp16})")
    return save_path


def load_trained_model(name, checkpoint=CHECKPOINT):
    load_path = f"{MODEL_SAVE_DIR}/{name}_state_dict.pt"
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(CLASS_ORDER), id2label=id2label, label2id=label2id
    )
    state_dict = torch.load(load_path, map_location=DEVICE)
    model_dtype = next(model.parameters()).dtype
    state_dict = {k: (v.to(model_dtype) if torch.is_tensor(v) else v) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    print(f"Đã tải lại '{name}' từ {load_path}")
    return model


def clean_text_for_bert(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_text_for_match(t):
    t = str(t)
    t = t.replace('\r\n', '\n').replace('\r', '\n')
    t = re.sub(r'\s+', ' ', t)
    return t.strip()


def get_confidence_group(scores):
    """Trả về (group_array, threshold, reason_string).
    Mặc định chia theo median. Nếu median trùng giá trị lớn nhất của cột
    (trường hợp DEPTWEET, nơi hơn một nửa mẫu đạt confidence_score = 1.0),
    median split sẽ suy biến. Khi đó chia theo score == max (high) vs còn lại (low)."""
    scores = np.asarray(scores, dtype=float)
    median = np.median(scores)
    if median == scores.max():
        group = np.where(scores == scores.max(), "high", "low")
        reason = (f"median ({median}) trùng giá trị trần -> chia theo "
                  f"score == {scores.max()} (high) vs còn lại (low)")
    else:
        group = np.where(scores < median, "low", "high")
        reason = f"chia theo median = {median:.3f}"
    return group, median, reason


def add_confidence_group(d, score_col):
    d = d.copy()
    group, thr, reason = get_confidence_group(d[score_col].values)
    d["conf_group"] = group
    d["conf_score_used"] = d[score_col]
    print(f"--- {score_col} ---")
    print(f"Lý do chọn ngưỡng: {reason}")
    print(d["conf_group"].value_counts())
    return d, thr


def plot_confusion_matrix(oof_df, name):
    cm = confusion_matrix(oof_df["true_label"], oof_df["pred_label"], labels=CLASS_ORDER)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_ORDER))); ax.set_xticklabels(CLASS_ORDER, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_ORDER))); ax.set_yticklabels(CLASS_ORDER)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix ({name})")
    for i in range(len(CLASS_ORDER)):
        for j in range(len(CLASS_ORDER)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/confusion_{name.lower().replace(' ', '_')}.png", dpi=150)
    plt.show()
    return cm

In [ ]:
# =============================================================================
# CELL 2 — Đọc DEPTWEET từ Google Sheet, merge, làm sạch
# =============================================================================
from google.colab import auth
import gspread
from google.auth import default
import time
from gspread.exceptions import APIError

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
RETRYABLE_STATUS = {429, 500, 502, 503, 504}

ROOT_SHEET_NAME = "deptweet_dataset_root"
APIFY_SHEET_NAME = "deptweet_dataset_apify"


def load_sheet_as_str_df(sheet_name, max_retries=5, base_delay=2.0):
    for attempt in range(1, max_retries + 1):
        try:
            ws = gc.open(sheet_name).get_worksheet(0)
            raw = ws.get_all_values()
            return pd.DataFrame(raw[1:], columns=raw[0])
        except APIError as e:
            status = e.response.status_code if getattr(e, "response", None) else None
            if status in RETRYABLE_STATUS and attempt < max_retries:
                time.sleep(base_delay * (2 ** (attempt - 1)))
                continue
            raise
    raise RuntimeError(f"Không đọc được sheet '{sheet_name}'.")


def fix_scientific_id(df, col):
    df[col] = df[col].astype(str).str.strip()
    bad = df[col].str.contains(r"e\+|E\+", regex=True, na=False)
    if bad.sum() > 0:
        print(f"  Loại {bad.sum()} dòng '{col}' bị mất precision (dạng khoa học).")
        df = df.loc[~bad].reset_index(drop=True)
    return df


root_df = fix_scientific_id(load_sheet_as_str_df(ROOT_SHEET_NAME), "tweet_id")
apify_df = fix_scientific_id(load_sheet_as_str_df(APIFY_SHEET_NAME), "id")

root_df["confidence_score"] = pd.to_numeric(
    root_df["confidence_score"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

merged_df = apify_df.merge(
    root_df[["tweet_id", "target", "label", "confidence_score"]],
    left_on="id", right_on="tweet_id", how="inner"
).drop(columns=["tweet_id"])
merged_df = merged_df.sort_values("confidence_score", ascending=False) \
                      .drop_duplicates(subset=["id"], keep="first")

df = merged_df[merged_df["lang"] == "en"].copy()
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"])
df["target"] = df["target"].astype(int)
df["label"] = pd.Categorical(df["label"], categories=CLASS_ORDER, ordered=True)
df = df.dropna(subset=["label"]).reset_index(drop=True)
df["label_id"] = df["label"].map(label2id).astype(int)
df["text_bert"] = df["text"].apply(clean_text_for_bert)
df = df[df["text_bert"].str.len() > 0].reset_index(drop=True)

print(f"\nDEPTWEET sau làm sạch: {len(df)} dòng")
print(df["label"].value_counts().reindex(CLASS_ORDER))

In [ ]:
# =============================================================================
# CELL 3 — E1: Xác thực mẫu tái thu hồi (χ² goodness-of-fit + binomial cho severe)
# =============================================================================
ORIGINAL_PROPORTIONS = {
    "non-depressed": 0.8062, "mild": 0.1304, "moderate": 0.0450, "severe": 0.0184,
}

N_total = len(df)
observed_counts = df["label"].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
expected_counts = pd.Series({cls: ORIGINAL_PROPORTIONS[cls] * N_total for cls in CLASS_ORDER})

print("=== Bảng I — Quan sát vs Kỳ vọng (số đếm) ===")
print(pd.DataFrame({"observed": observed_counts, "expected": expected_counts.round(1)}))

chi2_stat, chi2_p = chisquare(f_obs=observed_counts.values, f_exp=expected_counts.values)
print(f"\nχ² = {chi2_stat:.2f}, df = {len(CLASS_ORDER)-1}, p = {chi2_p:.3f}")

obs_severe = int(observed_counts["severe"])
exp_severe_p = ORIGINAL_PROPORTIONS["severe"]
binom_result = binomtest(obs_severe, n=N_total, p=exp_severe_p, alternative="less")
expected_severe_count = exp_severe_p * N_total
relative_shortfall = (obs_severe - expected_severe_count) / expected_severe_count * 100

print(f"\n=== Severe: binomial một phía ===")
print(f"Quan sát: {obs_severe} | Kỳ vọng: {expected_severe_count:.1f} | "
      f"Hao hụt tương đối: {relative_shortfall:.1f}%")
print(f"p-value = {binom_result.pvalue:.3f}")

In [ ]:
# =============================================================================
# CELL 4 — Dataset class
# =============================================================================
class DepressionDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# =============================================================================
# CELL 5 — Huấn luyện: 5-fold Stratified CV × nhiều seed, thu OOF predictions
# =============================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "qwk": cohen_kappa_score(labels, preds, weights="quadratic"),
        "mae_ordinal": mean_absolute_error(labels, preds),
    }


class BestModelTrackerCallback(TrainerCallback):
    def __init__(self, patience=2, metric_name="eval_macro_f1"):
        self.patience = patience
        self.metric_name = metric_name
        self.best_metric = None
        self.best_state_dict = None
        self.wait = 0

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or model is None:
            return control
        current = metrics.get(self.metric_name)
        if current is None:
            return control
        if self.best_metric is None or current > self.best_metric:
            self.best_metric = current
            self.best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                control.should_training_stop = True
        return control

    def restore_best(self, model):
        if self.best_state_dict is not None:
            model.load_state_dict(self.best_state_dict)
        return model


def build_training_args(run_dir, seed, epochs, n_train_samples):
    valid_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    steps_per_epoch = max(1, n_train_samples // BATCH_SIZE)
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)

    kwargs = dict(
        output_dir=run_dir, learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=epochs, weight_decay=WEIGHT_DECAY, warmup_steps=warmup_steps,
        save_strategy="no", logging_steps=50, report_to="none", seed=seed,
    )
    if "eval_strategy" in valid_params:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in valid_params:
        kwargs["evaluation_strategy"] = "epoch"

    kwargs = {k: v for k, v in kwargs.items() if k in valid_params}
    return TrainingArguments(**kwargs)


def run_cv_training(data, checkpoint, seeds=SEEDS, n_folds=N_FOLDS,
                     epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE, ckpt_tag="run",
                     save_mode=None):
    if save_mode is None:
        save_mode = SAVE_MODE

    texts_all = data["text_bert"].values
    labels_all = data["label_id"].values
    has_true_conf = "confidence_score" in data.columns
    conf_all = data["confidence_score"].values if has_true_conf else np.full(len(data), np.nan)
    n = len(data)

    oof_probs_accum = np.zeros((n, len(CLASS_ORDER)))
    oof_seen_count = np.zeros(n)
    fold_metrics_records = []

    best_per_seed = {}
    best_overall = {"macro_f1": -1, "state_dict": None, "seed": None, "fold": None}

    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(texts_all, labels_all)):
            trainval_labels = labels_all[trainval_idx]
            train_idx, val_idx = train_test_split(
                trainval_idx, test_size=0.10, stratify=trainval_labels, random_state=seed
            )

            train_ds = DepressionDataset(texts_all[train_idx], labels_all[train_idx], tokenizer)
            val_ds = DepressionDataset(texts_all[val_idx], labels_all[val_idx], tokenizer)
            test_ds = DepressionDataset(texts_all[test_idx], labels_all[test_idx], tokenizer)

            model = AutoModelForSequenceClassification.from_pretrained(
                checkpoint, num_labels=len(CLASS_ORDER), id2label=id2label, label2id=label2id
            ).to(DEVICE)

            run_dir = f"{CKPT_DIR}/{ckpt_tag}_seed{seed}_fold{fold_idx}"
            args = build_training_args(run_dir, seed, epochs, n_train_samples=len(train_idx))
            best_tracker = BestModelTrackerCallback(patience=patience)

            trainer = Trainer(
                model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                compute_metrics=compute_metrics, callbacks=[best_tracker],
            )

            print(f"\n--- [{ckpt_tag}] Seed {seed} | Fold {fold_idx} | "
                  f"train={len(train_idx)} val={len(val_idx)} test={len(test_idx)} ---")
            trainer.train()
            model = best_tracker.restore_best(model)

            test_output = trainer.predict(test_ds)
            test_probs = torch.softmax(torch.tensor(test_output.predictions), dim=1).numpy()
            test_preds = np.argmax(test_probs, axis=1)
            test_labels = labels_all[test_idx]
            fold_macro_f1 = f1_score(test_labels, test_preds, average="macro")

            fold_metrics_records.append({
                "seed": seed, "fold": fold_idx,
                "macro_f1": fold_macro_f1,
                "qwk": cohen_kappa_score(test_labels, test_preds, weights="quadratic"),
                "mae_ordinal": mean_absolute_error(test_labels, test_preds),
                **{f"f1_{cls}": f1_score(test_labels, test_preds, labels=[label2id[cls]], average="macro")
                   for cls in CLASS_ORDER}
            })

            if save_mode in ("best_per_seed", "single_best"):
                cpu_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                if seed not in best_per_seed or fold_macro_f1 > best_per_seed[seed][0]:
                    best_per_seed[seed] = (fold_macro_f1, cpu_state, fold_idx)
                if fold_macro_f1 > best_overall["macro_f1"]:
                    best_overall = {"macro_f1": fold_macro_f1, "state_dict": cpu_state,
                                     "seed": seed, "fold": fold_idx}

            oof_probs_accum[test_idx] += test_probs
            oof_seen_count[test_idx] += 1

            del model, trainer, best_tracker
            torch.cuda.empty_cache()
            shutil.rmtree(run_dir, ignore_errors=True)

    if save_mode == "best_per_seed":
        for seed, (f1_val, state_dict, fold_idx) in best_per_seed.items():
            fake_model = AutoModelForSequenceClassification.from_pretrained(
                checkpoint, num_labels=len(CLASS_ORDER), id2label=id2label, label2id=label2id)
            fake_model.load_state_dict(state_dict)
            save_trained_model(fake_model, f"{ckpt_tag}_seed{seed}_best")
            print(f"  (seed {seed}: model tốt nhất là fold {fold_idx}, macro_f1={f1_val:.3f})")
            del fake_model
    elif save_mode == "single_best":
        fake_model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint, num_labels=len(CLASS_ORDER), id2label=id2label, label2id=label2id)
        fake_model.load_state_dict(best_overall["state_dict"])
        save_trained_model(fake_model, f"{ckpt_tag}_single_best")
        print(f"  (model tốt nhất toàn bộ: seed {best_overall['seed']}, fold {best_overall['fold']}, "
              f"macro_f1={best_overall['macro_f1']:.3f})")
        del fake_model

    oof_probs_mean = oof_probs_accum / oof_seen_count[:, None]
    oof_preds_final = np.argmax(oof_probs_mean, axis=1)
    pred_confidence = oof_probs_mean.max(axis=1)

    oof_results_df = pd.DataFrame({
        "text": texts_all,
        "true_label_id": labels_all,
        "true_label": [id2label[l] for l in labels_all],
        "confidence_score": conf_all,
        "pred_confidence": pred_confidence,
        "pred_label_id": oof_preds_final,
        "pred_label": [id2label[l] for l in oof_preds_final],
        "correct": (oof_preds_final == labels_all),
    })
    return oof_results_df, pd.DataFrame(fold_metrics_records)

In [ ]:
# =============================================================================
# CELL 6 — Load lại kết quả DEPTWEET đã train, KHÔNG train lại (nếu có sẵn)
# =============================================================================
OOF_DEPTWEET_PATH = f"{BASE_DIR}/oof_predictions_deptweet.csv"
METRICS_DEPTWEET_PATH = f"{BASE_DIR}/fold_metrics_deptweet.csv"

if os.path.exists(OOF_DEPTWEET_PATH) and os.path.exists(METRICS_DEPTWEET_PATH):
    oof_distilbert_df = pd.read_csv(OOF_DEPTWEET_PATH)
    metrics_distilbert_df = pd.read_csv(METRICS_DEPTWEET_PATH)
    if oof_distilbert_df["correct"].dtype != bool:
        oof_distilbert_df["correct"] = oof_distilbert_df["correct"].astype(str).str.lower().map(
            {"true": True, "false": False}
        ).astype(bool)
    print(f"Đã LOAD LẠI OOF DEPTWEET từ Drive ({len(oof_distilbert_df)} dòng) -- KHÔNG train lại.")
    print(f"Đã LOAD LẠI fold metrics ({len(metrics_distilbert_df)} dòng).")
else:
    print("Chưa thấy file OOF/metrics DEPTWEET trên Drive -> train lại từ đầu.")
    oof_distilbert_df, metrics_distilbert_df = run_cv_training(
        df, checkpoint=CHECKPOINT, seeds=SEEDS, ckpt_tag="distilbert_deptweet"
    )
    oof_distilbert_df.to_csv(OOF_DEPTWEET_PATH, index=False)
    metrics_distilbert_df.to_csv(METRICS_DEPTWEET_PATH, index=False)
    print(f"Đã train xong và lưu OOF DEPTWEET ({len(oof_distilbert_df)} dòng).")

print("\n=== Bảng II (DEPTWEET) — DistilBERT, mean ± std qua fold/seed ===")
summary_deptweet = metrics_distilbert_df[["macro_f1", "qwk", "mae_ordinal"] +
                                          [f"f1_{c}" for c in CLASS_ORDER]].agg(["mean", "std"])
print(summary_deptweet.round(3))

In [ ]:
# =============================================================================
# CELL 6a — Mô hình NỀN TF-IDF + Logistic Regression
# =============================================================================
def run_cv_tfidf_lr_baseline(data, seeds=SEEDS, n_folds=N_FOLDS, ckpt_tag="tfidf_lr_deptweet"):
    texts_all = data["text_bert"].values
    labels_all = data["label_id"].values
    n = len(data)
    fold_metrics_records = []
    oof_probs_accum = np.zeros((n, len(CLASS_ORDER)))
    oof_seen_count = np.zeros(n)
    best_per_seed = {}

    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(texts_all, labels_all)):
            vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
            X_train = vec.fit_transform(texts_all[train_idx])
            X_test = vec.transform(texts_all[test_idx])
            clf = LogisticRegression(max_iter=2000, class_weight="balanced")
            clf.fit(X_train, labels_all[train_idx])
            probs = clf.predict_proba(X_test)
            preds = np.argmax(probs, axis=1)
            test_labels = labels_all[test_idx]
            fold_macro_f1 = f1_score(test_labels, preds, average="macro")
            if seed not in best_per_seed or fold_macro_f1 > best_per_seed[seed][0]:
                best_per_seed[seed] = (fold_macro_f1, vec, clf)
            fold_metrics_records.append({
                "seed": seed, "fold": fold_idx, "macro_f1": fold_macro_f1,
                "qwk": cohen_kappa_score(test_labels, preds, weights="quadratic"),
                "mae_ordinal": mean_absolute_error(test_labels, preds),
                **{f"f1_{cls}": f1_score(test_labels, preds, labels=[label2id[cls]], average="macro")
                   for cls in CLASS_ORDER}
            })
            oof_probs_accum[test_idx] += probs
            oof_seen_count[test_idx] += 1

    for seed, (f1_val, vec, clf) in best_per_seed.items():
        save_path = f"{MODEL_SAVE_DIR}/{ckpt_tag}_seed{seed}_best.joblib"
        joblib.dump({"vectorizer": vec, "classifier": clf}, save_path)
        print(f"  [lưu model] {ckpt_tag}_seed{seed}_best -> {save_path} (macro_f1={f1_val:.3f})")

    oof_preds_final = np.argmax(oof_probs_accum / oof_seen_count[:, None], axis=1)
    return pd.DataFrame(fold_metrics_records), oof_preds_final


METRICS_TFIDF_PATH = f"{BASE_DIR}/fold_metrics_tfidf_baseline.csv"
if os.path.exists(METRICS_TFIDF_PATH):
    metrics_tfidf_df = pd.read_csv(METRICS_TFIDF_PATH)
    print(f"Đã LOAD LẠI metrics TF-IDF+LR ({len(metrics_tfidf_df)} dòng) -- KHÔNG train lại.")
else:
    metrics_tfidf_df, oof_preds_tfidf = run_cv_tfidf_lr_baseline(df)
    metrics_tfidf_df.to_csv(METRICS_TFIDF_PATH, index=False)
    print(f"Đã train xong TF-IDF+LR, lưu metrics ({len(metrics_tfidf_df)} dòng).")

print("\n=== Bảng II — Mô hình nền TF-IDF + Logistic Regression (DEPTWEET) ===")
summary_tfidf = metrics_tfidf_df[["macro_f1", "qwk", "mae_ordinal"] +
                                  [f"f1_{c}" for c in CLASS_ORDER]].agg(["mean", "std"])
print(summary_tfidf.round(3))

# =============================================================================
# CELL 6b — Hình 1: Confusion matrix 4x4 (DEPTWEET)
# =============================================================================
cm_deptweet = plot_confusion_matrix(oof_distilbert_df, "DEPTWEET")

In [ ]:
# =============================================================================
# CELL 7a — Merge DepSeverity với Dreaddit gốc (đọc thẳng từ Drive)
# =============================================================================
DEPSEV_MERGED_PATH = f"{BASE_DIR}/depseverity_merged_with_confidence.csv"

if os.path.exists(DEPSEV_MERGED_PATH):
    depseverity_df = pd.read_csv(DEPSEV_MERGED_PATH)
    depseverity_df["label"] = pd.Categorical(depseverity_df["label"], categories=CLASS_ORDER, ordered=True)
    print(f"Đã LOAD LẠI depseverity_df đã merge ({len(depseverity_df)} dòng) -- KHÔNG merge lại.")
else:
    dreaddit_train = pd.read_csv(f"{BASE_DIR}/dreaddit-train.csv")
    dreaddit_test = pd.read_csv(f"{BASE_DIR}/dreaddit-test.csv")
    print(f"Dreaddit train: {len(dreaddit_train)} dòng | cột: {list(dreaddit_train.columns)}")
    print(f"Dreaddit test:  {len(dreaddit_test)} dòng | cột: {list(dreaddit_test.columns)}")

    dreaddit_df = pd.concat([dreaddit_train, dreaddit_test], ignore_index=True)
    print(f"\nDreaddit gộp train+test: {len(dreaddit_df)} dòng")

    assert "confidence" in dreaddit_df.columns, "Không thấy cột 'confidence' trong file Dreaddit."
    assert "text" in dreaddit_df.columns, "Không thấy cột 'text' trong file Dreaddit."

    dreaddit_df["text_norm"] = dreaddit_df["text"].apply(normalize_text_for_match)
    n_dup = dreaddit_df["text_norm"].duplicated().sum()
    if n_dup > 0:
        print(f"CẢNH BÁO: {n_dup} dòng trùng text_norm nội bộ Dreaddit -> giữ dòng đầu.")
        dreaddit_df = dreaddit_df.drop_duplicates(subset=["text_norm"], keep="first")

    DREADDIT_META_COLS = ["confidence", "subreddit", "post_id", "social_timestamp",
                           "social_karma", "social_upvote_ratio", "social_num_comments"]
    DREADDIT_META_COLS = [c for c in DREADDIT_META_COLS if c in dreaddit_df.columns]
    print(f"Cột metadata Dreaddit sẽ merge sang DepSeverity: {DREADDIT_META_COLS}")

    DEPSEV_LABEL_MAP = {"minimum": "non-depressed", "mild": "mild",
                         "moderate": "moderate", "severe": "severe"}

    d = pd.read_csv(f"{BASE_DIR}/DepSeverity.csv")[["text", "label"]].dropna()
    d["label_raw"] = d["label"].astype(str).str.strip().str.lower()
    print("=== Phân bố nhãn GỐC trong DepSeverity (trước khi map) ===")
    print(d["label_raw"].value_counts())
    d["label"] = d["label_raw"].map(DEPSEV_LABEL_MAP)
    n_unmapped = d["label"].isna().sum()
    if n_unmapped > 0:
        print(f"\nCẢNH BÁO: {n_unmapped} dòng label không map được -- đã loại.")
    d = d.dropna(subset=["label"]).reset_index(drop=True)

    d["text_norm"] = d["text"].apply(normalize_text_for_match)
    n_raw = len(d)
    d = d.drop_duplicates(subset=["text_norm"]).reset_index(drop=True)
    print(f"\nDepSeverity: {n_raw} dòng gốc -> {len(d)} sau loại trùng nội bộ")

    merged = d.merge(dreaddit_df[["text_norm"] + DREADDIT_META_COLS], on="text_norm", how="left")
    n_matched = merged["confidence"].notna().sum()
    print(f"\nKhớp với Dreaddit: {n_matched}/{len(merged)} = {n_matched/len(merged)*100:.1f}%")

    n_unmatched = merged["confidence"].isna().sum()
    if n_unmatched > 0:
        print(f"  -> Loại {n_unmatched} dòng KHÔNG khớp Dreaddit (không có confidence thật).")
        merged = merged.dropna(subset=["confidence"]).reset_index(drop=True)

    n_zero = (merged["confidence"] == 0).sum()
    print(f"  -> Loại {n_zero} dòng có confidence = 0 (dữ liệu thiếu theo định nghĩa).")
    merged = merged[merged["confidence"] > 0].reset_index(drop=True)

    merged["label"] = pd.Categorical(merged["label"], categories=CLASS_ORDER, ordered=True)
    merged["label_id"] = merged["label"].map(label2id).astype(int)
    merged["text_bert"] = merged["text"].apply(clean_text_for_bert)
    merged = merged[merged["text_bert"].str.len() > 0].reset_index(drop=True)
    merged = merged.rename(columns={"confidence": "confidence_score"})

    depseverity_df = merged[["text", "text_bert", "label", "label_id", "confidence_score"] +
                             [c for c in DREADDIT_META_COLS if c != "confidence"]]
    depseverity_df.to_csv(DEPSEV_MERGED_PATH, index=False)
    print(f"\nĐã lưu depseverity_df đã merge ({len(depseverity_df)} dòng) -> {DEPSEV_MERGED_PATH}")

print(f"\nDepSeverity sẵn sàng dùng: {len(depseverity_df)} dòng")
print(depseverity_df["label"].value_counts().reindex(CLASS_ORDER))

In [ ]:
# =============================================================================
# CELL 8 — Train DepSeverity (load-or-train, dùng confidence THẬT từ Dreaddit)
# =============================================================================
OOF_DEPSEV_PATH = f"{BASE_DIR}/oof_predictions_depseverity.csv"
METRICS_DEPSEV_PATH = f"{BASE_DIR}/fold_metrics_depsev.csv"

if os.path.exists(OOF_DEPSEV_PATH) and os.path.exists(METRICS_DEPSEV_PATH):
    oof_depsev_df = pd.read_csv(OOF_DEPSEV_PATH)
    metrics_depsev_df = pd.read_csv(METRICS_DEPSEV_PATH)
    if oof_depsev_df["correct"].dtype != bool:
        oof_depsev_df["correct"] = oof_depsev_df["correct"].astype(str).str.lower().map(
            {"true": True, "false": False}).astype(bool)
    print(f"Đã LOAD LẠI OOF DepSeverity ({len(oof_depsev_df)} dòng) -- KHÔNG train lại.")
else:
    print("Chưa có OOF DepSeverity -> train lại từ đầu (tốn thời gian GPU).")
    oof_depsev_df, metrics_depsev_df = run_cv_training(
        depseverity_df, checkpoint=CHECKPOINT, seeds=SEEDS, ckpt_tag="distilbert_depsev"
    )
    oof_depsev_df.to_csv(OOF_DEPSEV_PATH, index=False)
    metrics_depsev_df.to_csv(METRICS_DEPSEV_PATH, index=False)
    print(f"Đã train xong DepSeverity, lưu OOF ({len(oof_depsev_df)} dòng).")

summary_depsev = metrics_depsev_df[["macro_f1", "qwk", "mae_ordinal"] +
                                    [f"f1_{c}" for c in CLASS_ORDER]].agg(["mean", "std"])

In [ ]:
# =============================================================================
# CELL 8b — E6: Confusion matrix + per-class metrics trên DepSeverity
# =============================================================================
cm_depsev = plot_confusion_matrix(oof_depsev_df, "DepSeverity")

print("\n=== E6 — Bảng per-class (DepSeverity) ===")
print(summary_depsev.round(3))

print("\n=== So sánh nhanh: per-class F1 DEPTWEET vs DepSeverity ===")
compare_e6 = pd.DataFrame({
    "DEPTWEET_f1_mean": summary_deptweet.loc["mean", [f"f1_{c}" for c in CLASS_ORDER]].values,
    "DepSeverity_f1_mean": summary_depsev.loc["mean", [f"f1_{c}" for c in CLASS_ORDER]].values,
}, index=CLASS_ORDER)
print(compare_e6.round(3))
compare_e6.to_csv(f"{BASE_DIR}/e6_crossdataset_perclass_f1.csv")

In [ ]:
# =============================================================================
# CELL 9 — Chia nhóm confidence (dùng confidence_score thật cho cả 2 dataset)
# =============================================================================
oof_distilbert_df, thr_deptweet = add_confidence_group(oof_distilbert_df, "confidence_score")
oof_depsev_df, thr_depsev = add_confidence_group(oof_depsev_df, "confidence_score")

In [ ]:
# =============================================================================
# CELL 10 — E2: Bảng III (gộp vs phân tầng) + tương quan confidence-error
# =============================================================================
def table_III_and_r(d, name):
    d = d.copy()
    d["error"] = (~d["correct"]).astype(int)

    pooled = d.groupby("conf_group")["error"].mean().rename("error_rate_pooled")
    stratified = d.groupby(["true_label", "conf_group"], observed=True)["error"].mean().unstack("conf_group")

    r_pooled, p_pooled = pearsonr(d["conf_score_used"], d["error"])
    r_by_class = {}
    for cls in CLASS_ORDER:
        sub = d[d["true_label"] == cls]
        if sub["error"].nunique() > 1 and len(sub) > 2:
            r_by_class[cls] = pearsonr(sub["conf_score_used"], sub["error"])
        else:
            r_by_class[cls] = (np.nan, np.nan)

    print(f"\n=== Bảng III — {name} ===")
    print("-- Gộp chung 4 lớp --")
    print(pooled.round(4))
    print(f"r(confidence, error) gộp chung = {r_pooled:.3f} (p={p_pooled:.3f})")
    print("\n-- Phân tầng theo lớp thật --")
    print(stratified.round(4))
    print("\nr theo từng lớp:")
    for cls, (r, p) in r_by_class.items():
        print(f"  {cls}: r={r:.3f} p={p:.3f}" if not np.isnan(r) else f"  {cls}: không đủ biến thiên")
    return pooled, stratified, r_pooled, r_by_class

table_III_depsev = table_III_and_r(
    oof_depsev_df, "DepSeverity (confidence thật từ Dreaddit)"
)
table_III_deptweet = table_III_and_r(
    oof_distilbert_df, "DEPTWEET (confidence_score thật)"
)

In [ ]:
# =============================================================================
# CELL 11 (FIXED)
# =============================================================================
def plot_error_curve(d, name, n_bins=5, use_conf_group=False):
    d = d.copy()
    d["error"] = (~d["correct"]).astype(int)

    if use_conf_group:
        d["conf_bin"] = d["conf_group"]
        bin_labels = ["low", "high"]
        xtick_labels = ["Low", "High"]
    else:
        d["conf_bin"] = pd.qcut(d["conf_score_used"], q=n_bins, duplicates="drop")
        bin_labels = None
        xtick_labels = None

    fig, ax = plt.subplots(figsize=(7, 5))
    for cls in CLASS_ORDER:
        sub = d[d["true_label"] == cls]
        if len(sub) == 0:
            continue
        curve = sub.groupby("conf_bin", observed=True)["error"].mean()
        if bin_labels is not None:
            curve = curve.reindex(bin_labels)
        ax.plot(range(len(curve)), curve.values, marker="o", label=cls.capitalize())

    ax.set_xlabel("Confidence group (low → high)" if use_conf_group else "Nhóm confidence (thấp → cao)")
    ax.set_ylabel("Error rate" if use_conf_group else "Tỷ lệ lỗi")
    ax.set_title(name)
    if xtick_labels is not None:
        ax.set_xticks(range(len(xtick_labels)))
        ax.set_xticklabels(xtick_labels)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/errorcurve_{name.lower().replace(' ', '_')}.png", dpi=150)
    plt.show()

plot_error_curve(oof_depsev_df, "E2 dry-run - DepSeverity")
plot_error_curve(oof_distilbert_df, "Error Rate by Confidence Group — DEPTWEET",
                  use_conf_group=True)

In [ ]:
# =============================================================================
# CELL 12 — E3: Train sạch / Test bẩn
# =============================================================================
def add_conf_group_raw(data, score_col="confidence_score"):
    data = data.copy()
    group, thr, reason = get_confidence_group(data[score_col].values)
    data["conf_group"] = group
    print(f"E3 dùng ngưỡng: {reason}")
    return data


def _train_one_model_e3(data, train_pool_idx, seed, epochs, patience, tag):
    tr_idx, val_idx = train_test_split(
        train_pool_idx, test_size=0.10,
        stratify=data.loc[train_pool_idx, "label_id"], random_state=seed
    )
    train_ds = DepressionDataset(data.loc[tr_idx, "text_bert"].values, data.loc[tr_idx, "label_id"].values, tokenizer)
    val_ds = DepressionDataset(data.loc[val_idx, "text_bert"].values, data.loc[val_idx, "label_id"].values, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=len(CLASS_ORDER), id2label=id2label, label2id=label2id
    ).to(DEVICE)
    run_dir = f"{CKPT_DIR}/e3_{tag}_seed{seed}"
    args = build_training_args(run_dir, seed, epochs, n_train_samples=len(tr_idx))
    tracker = BestModelTrackerCallback(patience=patience)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics, callbacks=[tracker])
    trainer.train()
    trainer.model = tracker.restore_best(trainer.model)

    save_trained_model(trainer.model, f"e3_{tag}_seed{seed}")

    shutil.rmtree(run_dir, ignore_errors=True)
    return trainer


def run_clean_dirty_experiment(data, seeds=SEEDS, epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE):
    data = add_conf_group_raw(data)
    records = []

    for seed in seeds:
        train_idx, test_idx = train_test_split(
            data.index, test_size=0.20, stratify=data["label_id"], random_state=seed
        )
        train_all = data.loc[train_idx]
        test_all = data.loc[test_idx]

        train_clean_idx = train_all[train_all["conf_group"] == "high"].index
        trainer_clean = _train_one_model_e3(data, train_clean_idx, seed, epochs, patience, tag="clean")

        trainer_full = _train_one_model_e3(data, train_all.index, seed, epochs, patience, tag="full")

        for model_name, trainer in [("clean_only", trainer_clean), ("full_data", trainer_full)]:
            for grp in ["high", "low"]:
                test_sub = test_all[test_all["conf_group"] == grp]
                if len(test_sub) == 0:
                    continue
                test_ds = DepressionDataset(test_sub["text_bert"].values, test_sub["label_id"].values, tokenizer)
                pred = trainer.predict(test_ds)
                preds = np.argmax(pred.predictions, axis=1)
                records.append({
                    "seed": seed, "test_group": grp, "train_mode": model_name,
                    "macro_f1": f1_score(test_sub["label_id"], preds, average="macro"),
                    "n_test": len(test_sub),
                })

        del trainer_clean, trainer_full
        torch.cuda.empty_cache()

    return pd.DataFrame(records)

E3_PATH = f"{BASE_DIR}/e3_clean_dirty.csv"
if os.path.exists(E3_PATH):
    e3_clean_dirty_df = pd.read_csv(E3_PATH)
    print(f"Đã LOAD LẠI E3 ({len(e3_clean_dirty_df)} dòng) -- KHÔNG train lại.")
else:
    e3_clean_dirty_df = run_clean_dirty_experiment(df)
    e3_clean_dirty_df.to_csv(E3_PATH, index=False)

print("\n=== E3 — Train sạch/toàn bộ x Test cao/thấp (Bảng IV, phần 1) ===")
print(e3_clean_dirty_df.groupby(["train_mode", "test_group"])["macro_f1"].agg(["mean", "std"]).round(3))

In [ ]:
# =============================================================================
# CELL 13 — E4: Đối chiếu trần đồng thuận
# =============================================================================
PUBLISHED_FLEISS_KAPPA = {"non-depressed": 0.44, "mild": 0.27, "moderate": 0.30, "severe": 0.45}

def model_label_kappa_by_class(oof_df):
    rows = []
    for cls in CLASS_ORDER:
        y_true_bin = (oof_df["true_label"] == cls).astype(int)
        y_pred_bin = (oof_df["pred_label"] == cls).astype(int)
        kappa = cohen_kappa_score(y_true_bin, y_pred_bin)
        rows.append({
            "class": cls,
            "model_label_kappa": round(kappa, 3),
            "annotator_fleiss_kappa": PUBLISHED_FLEISS_KAPPA[cls],
            "model_reaches_human_ceiling": kappa >= PUBLISHED_FLEISS_KAPPA[cls],
        })
    return pd.DataFrame(rows)

e4_table = model_label_kappa_by_class(oof_distilbert_df)
print("\n=== E4 — cột bổ sung cho Bảng II ===")
print(e4_table)
e4_table.to_csv(f"{BASE_DIR}/e4_kappa_comparison.csv", index=False)

In [ ]:
# =============================================================================
# CELL 14 (SỬA — chạy 3 seed để có mean ± std, và FIX lỗi thiếu set_seed
# khiến classifier head khởi tạo ngẫu nhiên không cố định được theo seed)
# =============================================================================
from transformers import set_seed

PHQ9_KEYWORDS = [
    "disinterest", "involved", "occupied", "pessimism", "reversion", "absorbed",
    "lifelessness", "bored", "enthusiasm", "engrossed", "worried", "apathy",
    "dejected", "dismayed", "dispirited", "demoralized", "grimmed",
    "misery", "grim", "downhearted", "low-spirited", "bleak", "desperate",
    "lost", "frustrated",
    "nap", "restless", "awake", "whole night", "bedtime",
    "weary", "fatigue", "fag", "fag out", "overtire", "overfatigued", "burned-out",
    "burnt-out", "exhausted", "dog-tired", "washed-out", "drained", "whacked",
    "aversion", "distaste", "loathing", "malformed", "bulimic", "puffy",
    "starve", "fat",
    "loser", "relapse", "downfall", "ruined", "flop", "dead-duck",
    "disappointment", "achiever", "misfire", "underdog", "falling-apart", "disgusted",
    "immersed", "decentralize", "deconcentrate", "scattered", "dispersed",
    "unsettled", "focus",
    "discontent", "ungratified", "unsatisfied", "stand-still", "refrained", "immobile",
    "trauma", "harm", "suffering", "anguish", "hemorrhage", "penetrating-trauma",
    "torment", "agony", "excruciate", "damaged", "gag", "suffocate", "self-destruction",
]
print(f"Số từ khoá PHQ-9 đang dùng: {len(PHQ9_KEYWORDS)} (phải = 89)")
kw_set = set(PHQ9_KEYWORDS)


def train_single_model_for_e5(data, seed, epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE):
    train_idx, test_idx = train_test_split(
        data.index, test_size=0.20, stratify=data["label_id"], random_state=seed
    )
    tr_idx, val_idx = train_test_split(
        train_idx, test_size=0.10, stratify=data.loc[train_idx, "label_id"], random_state=seed
    )
    train_ds = DepressionDataset(data.loc[tr_idx, "text_bert"].values, data.loc[tr_idx, "label_id"].values, tokenizer)
    val_ds = DepressionDataset(data.loc[val_idx, "text_bert"].values, data.loc[val_idx, "label_id"].values, tokenizer)

    # FIX: cố định seed TRƯỚC khi tạo model, vì classifier/pre_classifier
    # head bị khởi tạo ngẫu nhiên ngay tại from_pretrained() -- TrainingArguments(seed=...)
    # chỉ set seed BÊN TRONG Trainer, tức là SAU khi model đã tạo xong, nên không
    # cố định được phần này -> kết quả không tái lập được giữa các lần chạy.
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=len(CLASS_ORDER), id2label=id2label, label2id=label2id
    ).to(DEVICE)

    run_dir = f"{CKPT_DIR}/e5_single_seed{seed}"
    args = build_training_args(run_dir, seed, epochs, n_train_samples=len(tr_idx))
    tracker = BestModelTrackerCallback(patience=patience)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics, callbacks=[tracker])
    trainer.train()
    trainer.model = tracker.restore_best(trainer.model)

    save_trained_model(trainer.model, f"e5_full_model_seed{seed}")

    shutil.rmtree(run_dir, ignore_errors=True)

    test_df = data.loc[test_idx].reset_index(drop=True)
    train_pool_df = data.loc[train_idx].reset_index(drop=True)
    return trainer, test_df, train_pool_df


def mask_words(text, words_to_mask):
    tokens = text.split()
    clean = [t.lower().strip(".,!?") for t in tokens]
    mask_flags = [False] * len(tokens)
    for i, t in enumerate(clean):
        if t in words_to_mask:
            mask_flags[i] = True
    for i in range(len(clean) - 1):
        bigram = f"{clean[i]} {clean[i+1]}"
        if bigram in words_to_mask:
            mask_flags[i] = True
            mask_flags[i + 1] = True
    return " ".join("[MASK]" if mask_flags[i] else tok for i, tok in enumerate(tokens))


def build_masked_random(texts, seed):
    rng = np.random.default_rng(seed)
    out = []
    for t in texts:
        tokens = t.split()
        n_kw_hits = sum(1 for tok in tokens if tok.lower().strip(".,!?") in kw_set)
        if len(tokens) == 0 or n_kw_hits == 0:
            out.append(t)
            continue
        mask_idx = set(rng.choice(len(tokens), size=min(n_kw_hits, len(tokens)), replace=False))
        out.append(" ".join("[MASK]" if i in mask_idx else tok for i, tok in enumerate(tokens)))
    return out


def eval_texts_with_model(trainer, texts, labels):
    ds = DepressionDataset(texts, labels, tokenizer)
    pred = trainer.predict(ds)
    preds = np.argmax(pred.predictions, axis=1)
    return f1_score(labels, preds, average="macro")


def run_e5_one_seed(seed):
    e5_trainer, e5_test_df, e5_train_pool_df = train_single_model_for_e5(df, seed=seed)

    full_model_f1 = eval_texts_with_model(
        e5_trainer, e5_test_df["text_bert"].tolist(), e5_test_df["label_id"].values
    )

    vec = CountVectorizer(vocabulary=PHQ9_KEYWORDS, binary=True, ngram_range=(1, 2))
    X_kw_train = vec.fit_transform(e5_train_pool_df["text_bert"].str.lower())
    X_kw_test = vec.transform(e5_test_df["text_bert"].str.lower())
    clf_kw = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf_kw.fit(X_kw_train, e5_train_pool_df["label_id"])
    kw_preds = clf_kw.predict(X_kw_test)
    keyword_only_f1 = f1_score(e5_test_df["label_id"], kw_preds, average="macro")

    joblib.dump({"vectorizer": vec, "classifier": clf_kw},
                f"{MODEL_SAVE_DIR}/e5_keyword_only_lr_seed{seed}.joblib")

    masked_kw_texts = e5_test_df["text_bert"].apply(lambda t: mask_words(t, kw_set)).tolist()
    masked_random_texts = build_masked_random(e5_test_df["text_bert"].tolist(), seed=seed)

    f1_keyword_masked = eval_texts_with_model(e5_trainer, masked_kw_texts, e5_test_df["label_id"].values)
    f1_random_masked = eval_texts_with_model(e5_trainer, masked_random_texts, e5_test_df["label_id"].values)

    del e5_trainer
    torch.cuda.empty_cache()

    return {
        "seed": seed,
        "full_model_f1": full_model_f1,
        "keyword_only_f1": keyword_only_f1,
        "keyword_masked_f1": f1_keyword_masked,
        "random_masked_f1": f1_random_masked,
        "n_test": len(e5_test_df),
    }


E5_MULTISEED_PATH = f"{BASE_DIR}/e5_keyword_masking_summary_3seed.csv"

if os.path.exists(E5_MULTISEED_PATH):
    e5_records_df = pd.read_csv(E5_MULTISEED_PATH)
    print(f"Đã LOAD LẠI E5 3-seed ({len(e5_records_df)} dòng) -- KHÔNG train lại.")
else:
    e5_records = [run_e5_one_seed(seed) for seed in SEEDS]
    e5_records_df = pd.DataFrame(e5_records)
    e5_records_df.to_csv(E5_MULTISEED_PATH, index=False)
    print(f"Đã train xong E5 qua {len(SEEDS)} seed, lưu {E5_MULTISEED_PATH}.")

print("\n=== E5 — Kết quả từng seed ===")
print(e5_records_df.round(3))

e5_summary = e5_records_df[["full_model_f1", "keyword_only_f1",
                             "keyword_masked_f1", "random_masked_f1"]].agg(["mean", "std"])
print("\n=== E5 — mean ± std qua 3 seed (Bảng IV, phần 2) ===")
print(e5_summary.round(3))

e5_summary.to_csv(f"{BASE_DIR}/e5_keyword_masking_summary.csv")
print(f"\nĐã lưu {BASE_DIR}/e5_keyword_masking_summary.csv (bản mean±std, ghi đè bản 1-seed cũ).")

In [ ]:
# =============================================================================
# CELL 15 — Tổng hợp
# =============================================================================
print(f"""
TỔNG HỢP KẾT QUẢ (đối chiếu outline v2, mục 5 — Danh mục bảng và hình):
  Bảng I   (IV.C, E1)                        -> Cell 3
  Bảng II  (IV.D, E2+E4)  -- gồm CẢ mô hình nền TF-IDF+LR (IV.B)
                                              -> Cell 6 (DistilBERT) + Cell 6a (TF-IDF+LR) + Cell 13 (kappa)
  Bảng III (IV.D, E2 -- trung tâm)           -> Cell 10; ngưỡng confidence lấy từ Cell 9
  Bảng IV  (IV.D, E3+E5)                     -> Cell 12 + Cell 14
  Hình 1   (IV.D, confusion 4x4)             -> Cell 6b
  Hình 2   (IV.D, lỗi theo confidence)       -> Cell 11
  E6 (nhân rộng DepSeverity)                 -> Cell 8b

MODEL ĐÃ LƯU tại {MODEL_SAVE_DIR} (save_mode = "{SAVE_MODE}", fp16 = {SAVE_AS_FP16}):
  - distilbert_deptweet_seed{{s}}_best_state_dict.pt   (Cell 6, chính, 3 file)
  - distilbert_depsev_seed{{s}}_best_state_dict.pt     (Cell 8, E6, 3 file)
  - tfidf_lr_deptweet_seed{{s}}_best.joblib            (Cell 6a, mô hình nền, 3 file)
  - e3_clean_seed{{s}}_state_dict.pt / e3_full_seed{{s}}_state_dict.pt  (Cell 12, 6 file)
  - e5_full_model_seed42_state_dict.pt                (Cell 14, 1 file)
  - e5_keyword_only_lr.joblib                         (Cell 14, 1 file)

GHI CHÚ QUAN TRỌNG:
  - DepSeverity (Cell 7a) KHÔNG có confidence riêng. confidence_score của nó
    được MERGE lại từ Dreaddit gốc (Turcan & McKeown, 2019) qua text chuẩn hoá,
    vì DepSeverity là bản gán nhãn lại DSM-5 của đúng 3.553 đoạn văn trong Dreaddit.
    Đã xác nhận thêm: hai bộ row-aligned 100% theo thứ tự train+test (xem Cell 21).
  - Confidence này đo TỶ LỆ ĐỒNG THUẬN ANNOTATOR TRÊN NHÃN STRESS NHỊ PHÂN
    (nhiệm vụ gốc của Dreaddit), KHÔNG PHẢI trên nhãn mức độ trầm cảm.
  - Dòng confidence = 0 (không khớp định nghĩa "tỷ lệ đồng thuận") đã bị loại ở Cell 7a.
    Kiểm tra tính đồng đều của việc loại này giữa các lớp: xem Cell 22.
  - Ngưỡng confidence ở E2 (Cell 9) và E3 (Cell 12) vẫn dùng CHUNG hàm
    get_confidence_group(), đảm bảo nhất quán giữa hai thí nghiệm.
  - E4 dùng cohen_kappa_score one-vs-rest, là proxy xấp xỉ cho Fleiss' kappa gốc.
  - Cell 11 (Hình 2 DEPTWEET) và Cell 14 (E5) đã sửa 2 lỗi đã biết từ trước
    (xem comment tại từng cell).
  - Chạy Cell 16 sau cùng của phần train để đóng gói toàn bộ model thành 1 file .zip.
""")

In [ ]:
# =============================================================================
# CELL 16 — Đóng gói toàn bộ model đã lưu thành 1 file .zip duy nhất
# =============================================================================
ZIP_PATH = f"{BASE_DIR}/saved_models_bundle"

archive_path = shutil.make_archive(ZIP_PATH, "zip", MODEL_SAVE_DIR)
zip_size_mb = os.path.getsize(archive_path) / (1024 ** 2)
print(f"Đã đóng gói: {archive_path} ({zip_size_mb:.1f} MB)")

all_saved = glob.glob(f"{MODEL_SAVE_DIR}/*")
print(f"\nSố file model trong gói: {len(all_saved)}")
for f in all_saved:
    print(" -", os.path.basename(f))

print(f"""
Muốn dùng lại model sau này:
  import zipfile
  with zipfile.ZipFile("{archive_path}", "r") as z:
      z.extractall("{MODEL_SAVE_DIR}")
""")

In [ ]:
# =============================================================================
# CELL 17 — Đối chiếu giao thức phân tầng: TF-IDF+LR vs DistilBERT (Discussion §3)
# =============================================================================
OOF_TFIDF_FULL_PATH = f"{BASE_DIR}/oof_predictions_tfidf.csv"

if os.path.exists(OOF_TFIDF_FULL_PATH):
    oof_tfidf_df = pd.read_csv(OOF_TFIDF_FULL_PATH)
    if oof_tfidf_df["correct"].dtype != bool:
        oof_tfidf_df["correct"] = oof_tfidf_df["correct"].astype(str).str.lower().map(
            {"true": True, "false": False}).astype(bool)
    print(f"Đã LOAD LẠI OOF TF-IDF+LR đầy đủ ({len(oof_tfidf_df)} dòng) -- KHÔNG train lại.")
else:
    texts_all = df["text_bert"].values
    labels_all = df["label_id"].values
    conf_all = df["confidence_score"].values
    n = len(df)
    oof_probs_accum = np.zeros((n, len(CLASS_ORDER)))
    oof_seen_count = np.zeros(n)

    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(texts_all, labels_all)):
            vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
            X_train = vec.fit_transform(texts_all[train_idx])
            X_test = vec.transform(texts_all[test_idx])
            clf = LogisticRegression(max_iter=2000, class_weight="balanced")
            clf.fit(X_train, labels_all[train_idx])
            probs = clf.predict_proba(X_test)
            oof_probs_accum[test_idx] += probs
            oof_seen_count[test_idx] += 1
        print(f"  Seed {seed} xong.")

    oof_probs_mean = oof_probs_accum / oof_seen_count[:, None]
    oof_preds_final = np.argmax(oof_probs_mean, axis=1)

    oof_tfidf_df = pd.DataFrame({
        "text": texts_all, "true_label_id": labels_all,
        "true_label": [id2label[l] for l in labels_all],
        "confidence_score": conf_all,
        "pred_label_id": oof_preds_final,
        "pred_label": [id2label[l] for l in oof_preds_final],
        "correct": (oof_preds_final == labels_all),
    })
    oof_tfidf_df.to_csv(OOF_TFIDF_FULL_PATH, index=False)
    print("Đã train xong TF-IDF+LR OOF đầy đủ.")

oof_tfidf_df, thr_tfidf = add_confidence_group(oof_tfidf_df, "confidence_score")

table_III_tfidf = table_III_and_r(oof_tfidf_df, "DEPTWEET — TF-IDF+LR")
table_III_distilbert = table_III_and_r(oof_distilbert_df, "DEPTWEET — DistilBERT")


def summarize_pooled_vs_stratified(oof_df, model_name):
    d = oof_df.copy()
    d["error"] = (~d["correct"]).astype(int)
    pooled_hi = d[d["conf_group"] == "high"]["error"].mean()
    pooled_lo = d[d["conf_group"] == "low"]["error"].mean()
    strat = d.groupby(["true_label", "conf_group"], observed=True)["error"].mean().unstack("conf_group")
    return pd.Series({
        "model": model_name,
        "pooled_error_high": pooled_hi, "pooled_error_low": pooled_lo,
        "mild_error_high": strat.loc["mild", "high"], "mild_error_low": strat.loc["mild", "low"],
        "moderate_error_high": strat.loc["moderate", "high"], "moderate_error_low": strat.loc["moderate", "low"],
    })

protocol_comparison = pd.DataFrame([
    summarize_pooled_vs_stratified(oof_tfidf_df, "TF-IDF+LR"),
    summarize_pooled_vs_stratified(oof_distilbert_df, "DistilBERT"),
])
print("\n=== Bảng đối chiếu giao thức: TF-IDF+LR vs DistilBERT (Discussion §3) ===")
print(protocol_comparison.round(3))
protocol_comparison.to_csv(f"{BASE_DIR}/discussion_protocol_comparison.csv", index=False)

In [ ]:
# =============================================================================
# CELL 18 — Xuất PDF: error curve DepSeverity, error curve DEPTWEET (bản fix),
# confusion matrix DEPTWEET
# =============================================================================
def plot_error_curve_qcut_pdf(d, name, n_bins=5):
    d = d.copy()
    d["conf_bin"] = pd.qcut(d["confidence_score"], q=n_bins, duplicates="drop")
    fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
    for cls in CLASS_ORDER:
        sub = d[d["true_label"] == cls]
        if len(sub) == 0:
            continue
        curve = sub.groupby("conf_bin", observed=True)["error"].mean()
        ax.plot(range(len(curve)), curve.values, marker="o", label=cls)
    ax.set_xlabel("Nhóm confidence (thấp → cao)")
    ax.set_ylabel("Tỷ lệ lỗi")
    ax.set_title(name)
    ax.legend()
    plt.tight_layout()
    fname = f"{BASE_DIR}/errorcurve_{name.lower().replace(' ', '_')}.pdf"
    plt.savefig(fname, format="pdf", bbox_inches="tight", pad_inches=0.02)
    plt.show()
    print(f"Đã lưu: {fname}")

# FIX 1: đảm bảo oof_depsev_df có cột "error" trước khi vẽ
if "error" not in oof_depsev_df.columns:
    oof_depsev_df["error"] = (~oof_depsev_df["correct"]).astype(int)

plot_error_curve_qcut_pdf(oof_depsev_df, "E2 dry-run - DepSeverity")

# FIX 2: đảm bảo oof_distilbert_df có cột "error" TRƯỚC khi lấy "sub" trong vòng lặp
if "error" not in oof_distilbert_df.columns:
    oof_distilbert_df["error"] = (~oof_distilbert_df["correct"]).astype(int)

name_dt = "Error Rate by Confidence Group — DEPTWEET"
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
for cls in CLASS_ORDER:
    sub = oof_distilbert_df[oof_distilbert_df["true_label"] == cls]
    if len(sub) == 0:
        continue
    curve = sub.groupby("conf_group", observed=True)["error"].mean().reindex(["low", "high"])
    ax.plot(range(len(curve)), curve.values, marker="o", label=cls.capitalize())
ax.set_xlabel("Confidence group (low → high)")
ax.set_ylabel("Error rate")
ax.set_title(name_dt)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Low", "High"])
ax.legend()
plt.tight_layout()
fname_dt = f"{BASE_DIR}/errorcurve_{name_dt.lower().replace(' ', '_')}.pdf"
plt.savefig(fname_dt, format="pdf", bbox_inches="tight", pad_inches=0.02)
plt.show()
print(f"Đã lưu: {fname_dt}")


def plot_confusion_matrix_pdf(oof_df, name):
    cm = confusion_matrix(oof_df["true_label"], oof_df["pred_label"], labels=CLASS_ORDER)
    fig, ax = plt.subplots(figsize=(5.5, 5), dpi=300)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_ORDER))); ax.set_xticklabels(CLASS_ORDER, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_ORDER))); ax.set_yticklabels(CLASS_ORDER)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix ({name})")
    for i in range(len(CLASS_ORDER)):
        for j in range(len(CLASS_ORDER)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im)
    plt.tight_layout()
    fname = f"{BASE_DIR}/confusion_{name.lower().replace(' ', '_')}.pdf"
    plt.savefig(fname, format="pdf", bbox_inches="tight", pad_inches=0.02)
    plt.show()
    print(f"Đã lưu: {fname}")
    return cm

cm_deptweet_pdf = plot_confusion_matrix_pdf(oof_distilbert_df, "DEPTWEET")

In [ ]:
# =============================================================================
# CELL 19 — Xuất danh sách tweet_id + confidence_score + label (KHÔNG có text)
# =============================================================================
RELEASE_PATH = f"{BASE_DIR}/deptweet_tweetid_confidence_label.csv"
CANDIDATE_PATHS = [f"{BASE_DIR}/deptweet_dataset.csv"]

found_valid = False
for path in CANDIDATE_PATHS:
    if not os.path.exists(path):
        continue
    try:
        cand = pd.read_csv(path)
    except Exception as e:
        print(f"Không đọc được {path}: {e}")
        continue

    id_col = next((c for c in ["id", "tweet_id"] if c in cand.columns), None)
    has_conf = "confidence_score" in cand.columns
    has_label = "label" in cand.columns
    print(f"Kiểm tra {path}: {len(cand)} dòng | id_col={id_col} | "
          f"confidence_score={has_conf} | label={has_label}")

    if id_col and has_conf and has_label and len(cand) == 3359:
        print(f"  -> Dùng được file này ({path}).")
        release_df = cand[[id_col, "confidence_score", "label"]].rename(columns={id_col: "tweet_id"})
        found_valid = True
        break
    else:
        print(f"  -> KHÔNG khớp yêu cầu, bỏ qua.")

if not found_valid:
    print("\nKhông tìm thấy file merge sẵn hợp lệ -> dựng lại từ df hiện có trong session.")
    release_df = df[["id", "confidence_score", "label"]].rename(columns={"id": "tweet_id"})

release_df["tweet_id"] = release_df["tweet_id"].astype(str)
release_df = release_df.sort_values("tweet_id").reset_index(drop=True)

print(f"\n=== File công bố cuối cùng: {len(release_df)} dòng ===")
print(release_df.head(10))
print("\nSố giá trị confidence_score duy nhất:", release_df["confidence_score"].nunique())
print(release_df["confidence_score"].describe())

assert "text" not in release_df.columns
assert set(release_df.columns) == {"tweet_id", "confidence_score", "label"}

release_df.to_csv(RELEASE_PATH, index=False, float_format=None)
print(f"\nĐã lưu: {RELEASE_PATH}")

In [ ]:
# =============================================================================
# CELL 20 — E3 mở rộng: thêm điều kiện "full_data_matched" (size-matched)
# =============================================================================
def get_confidence_group_arr(scores):
    scores = np.asarray(scores, dtype=float)
    median = np.median(scores)
    if median == scores.max():
        return np.where(scores == scores.max(), "high", "low")
    return np.where(scores < median, "low", "high")


def _train_one_model_e3b(data_subset, seed, epochs, patience, tag):
    tr_idx, val_idx = train_test_split(
        data_subset.index, test_size=0.10,
        stratify=data_subset["label_id"], random_state=seed
    )
    train_ds = DepressionDataset(data_subset.loc[tr_idx, "text_bert"].values,
                                  data_subset.loc[tr_idx, "label_id"].values, tokenizer)
    val_ds = DepressionDataset(data_subset.loc[val_idx, "text_bert"].values,
                                data_subset.loc[val_idx, "label_id"].values, tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=len(CLASS_ORDER), id2label=id2label, label2id=label2id
    ).to(DEVICE)
    run_dir = f"{CKPT_DIR}/e3b_{tag}_seed{seed}"
    args = build_training_args(run_dir, seed, epochs, n_train_samples=len(tr_idx))
    tracker = BestModelTrackerCallback(patience=patience)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics, callbacks=[tracker])
    trainer.train()
    trainer.model = tracker.restore_best(trainer.model)
    shutil.rmtree(run_dir, ignore_errors=True)
    return trainer


def run_e3_size_matched(data, seeds=SEEDS, epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE):
    records = []
    for seed in seeds:
        train_idx, test_idx = train_test_split(
            data.index, test_size=0.20, stratify=data["label_id"], random_state=seed
        )
        train_all = data.loc[train_idx]
        test_all = data.loc[test_idx]

        train_clean = train_all[train_all["conf_group"] == "high"]
        n_clean = len(train_clean)
        print(f"\n[Seed {seed}] train_clean = {n_clean} dòng, train_all = {len(train_all)} dòng")

        train_full_matched = train_all.groupby("label_id", group_keys=False).apply(
            lambda g: g.sample(
                n=min(len(g), max(1, round(n_clean * len(g) / len(train_all)))),
                random_state=seed
            )
        )
        if len(train_full_matched) > n_clean:
            train_full_matched = train_full_matched.sample(n=n_clean, random_state=seed)
        elif len(train_full_matched) < n_clean:
            remaining = train_all.drop(train_full_matched.index)
            extra = remaining.sample(n=n_clean - len(train_full_matched), random_state=seed)
            train_full_matched = pd.concat([train_full_matched, extra])
        print(f"  train_full_matched = {len(train_full_matched)} dòng (khớp n_clean)")

        conditions = {"clean_only": train_clean, "full_data": train_all,
                       "full_data_matched": train_full_matched}

        for cond_name, train_subset in conditions.items():
            trainer = _train_one_model_e3b(train_subset, seed, epochs, patience, tag=cond_name)
            for grp in ["high", "low"]:
                test_sub = test_all[test_all["conf_group"] == grp]
                if len(test_sub) == 0:
                    continue
                test_ds = DepressionDataset(test_sub["text_bert"].values, test_sub["label_id"].values, tokenizer)
                pred = trainer.predict(test_ds)
                preds = np.argmax(pred.predictions, axis=1)
                records.append({
                    "seed": seed, "test_group": grp, "train_mode": cond_name,
                    "macro_f1": f1_score(test_sub["label_id"], preds, average="macro"),
                    "n_train": len(train_subset), "n_test": len(test_sub),
                })
            del trainer
            torch.cuda.empty_cache()
    return pd.DataFrame(records)


df_e3b = df.copy()
df_e3b["conf_group"] = get_confidence_group_arr(df_e3b["confidence_score"].values)

E3B_PATH = f"{BASE_DIR}/e3_size_matched.csv"
if os.path.exists(E3B_PATH):
    e3b_df = pd.read_csv(E3B_PATH)
    print(f"Đã LOAD LẠI E3 size-matched ({len(e3b_df)} dòng) -- KHÔNG train lại.")
else:
    e3b_df = run_e3_size_matched(df_e3b)
    e3b_df.to_csv(E3B_PATH, index=False)

print("\n=== Bảng IV mở rộng: 3 điều kiện train ===")
print(e3b_df.groupby(["train_mode", "test_group"])["macro_f1"].agg(["mean", "std"]).round(3))
print("\nCỡ tập train mỗi điều kiện (trung bình qua 3 seed):")
print(e3b_df.groupby("train_mode")["n_train"].mean().round(0))

In [ ]:
# =============================================================================
# CELL 21 — Kiểm tra DepSeverity có row-aligned với Dreaddit không
# =============================================================================
dep_check = pd.read_csv(f"{BASE_DIR}/DepSeverity.csv")
dreaddit_train_check = pd.read_csv(f"{BASE_DIR}/dreaddit-train.csv")
dreaddit_test_check = pd.read_csv(f"{BASE_DIR}/dreaddit-test.csv")

print(f"DepSeverity: {len(dep_check)} dòng")
print(f"Dreaddit train+test: {len(dreaddit_train_check) + len(dreaddit_test_check)} dòng")

for order_name, dreaddit_ordered in [
    ("train+test", pd.concat([dreaddit_train_check, dreaddit_test_check], ignore_index=True)),
    ("test+train", pd.concat([dreaddit_test_check, dreaddit_train_check], ignore_index=True)),
]:
    print(f"\n--- Thử thứ tự: {order_name} ---")
    if len(dep_check) != len(dreaddit_ordered):
        print(f"KHÁC độ dài -> không thể row-align.")
        continue
    dep_norm = dep_check["text"].apply(normalize_text_for_match)
    dr_norm = dreaddit_ordered["text"].apply(normalize_text_for_match)
    n_match_position = (dep_norm.values == dr_norm.values).sum()
    pct = n_match_position / len(dep_check) * 100
    print(f"Khớp đúng vị trí: {n_match_position}/{len(dep_check)} = {pct:.1f}%")
    if pct == 100.0:
        print(">>> HOÀN TOÀN row-aligned theo thứ tự này.")

In [ ]:
# =============================================================================
# CELL 22 — Kiểm tra loại 98 dòng confidence=0 có đồng đều giữa các lớp + sensitivity check
# =============================================================================
dreaddit_df_check = pd.concat([dreaddit_train_check, dreaddit_test_check], ignore_index=True)
dep_check2 = dep_check.reset_index(drop=True)
dreaddit_df_check = dreaddit_df_check.reset_index(drop=True)
combined = pd.concat([dep_check2[["text", "label"]], dreaddit_df_check[["confidence"]]], axis=1)

DEPSEV_LABEL_MAP2 = {"minimum": "non-depressed", "mild": "mild", "moderate": "moderate", "severe": "severe"}
combined["label_mapped"] = combined["label"].astype(str).str.strip().str.lower().map(DEPSEV_LABEL_MAP2)
combined = combined.dropna(subset=["label_mapped"])

combined["text_norm"] = combined["text"].apply(normalize_text_for_match)
combined = combined.drop_duplicates(subset=["text_norm"], keep="first").reset_index(drop=True)
print(f"Sau loại trùng nội bộ: {len(combined)} dòng (kỳ vọng 3532)")

combined["is_zero_conf"] = (combined["confidence"] == 0)

print("\n=== Số dòng bị loại (confidence=0) theo từng lớp ===")
summary_zero = combined.groupby("label_mapped")["is_zero_conf"].agg(["sum", "count"])
summary_zero["pct_dropped"] = (summary_zero["sum"] / summary_zero["count"] * 100).round(1)
summary_zero = summary_zero.reindex(CLASS_ORDER)
print(summary_zero)

contingency = pd.crosstab(combined["label_mapped"], combined["is_zero_conf"])
chi2_val, p_val, dof, expected = chi2_contingency(contingency)
print(f"\nχ² = {chi2_val:.4f}, df = {dof}, p = {p_val:.4f}")


# --- Sensitivity check: train lại DistilBERT GIỮ NGUYÊN 98 dòng confidence=0 ---
depsev_all_df = combined.copy()
depsev_all_df["label"] = pd.Categorical(depsev_all_df["label_mapped"], categories=CLASS_ORDER, ordered=True)
depsev_all_df["label_id"] = depsev_all_df["label"].map(label2id).astype(int)
depsev_all_df["text_bert"] = depsev_all_df["text"].apply(clean_text_for_bert)
depsev_all_df = depsev_all_df[depsev_all_df["text_bert"].str.len() > 0].reset_index(drop=True)
depsev_all_df = depsev_all_df.rename(columns={"confidence": "confidence_score"})
print(f"\ndepseverity_df (GIỮ confidence=0): {len(depsev_all_df)} dòng (kỳ vọng 3532)")
print(depsev_all_df["label"].value_counts().reindex(CLASS_ORDER))

SENS_PATH = f"{BASE_DIR}/oof_depseverity_sensitivity_keepzero.csv"
if os.path.exists(SENS_PATH):
    oof_sens_df = pd.read_csv(SENS_PATH)
    if oof_sens_df["correct"].dtype != bool:
        oof_sens_df["correct"] = oof_sens_df["correct"].astype(str).str.lower().map(
            {"true": True, "false": False}).astype(bool)
    print(f"Đã LOAD LẠI sensitivity check ({len(oof_sens_df)} dòng) -- KHÔNG train lại.")
else:
    oof_sens_df, _ = run_cv_training(
        depsev_all_df, checkpoint=CHECKPOINT, seeds=SEEDS, ckpt_tag="depsev_sensitivity",
        save_mode="none"
    )
    oof_sens_df.to_csv(SENS_PATH, index=False)
    print(f"Đã train xong sensitivity check, lưu {SENS_PATH}.")

oof_sens_df["conf_group"] = get_confidence_group_arr(oof_sens_df["confidence_score"].values)
oof_sens_df["error"] = (~oof_sens_df["correct"]).astype(int)

r_pooled_sens, p_pooled_sens = pearsonr(oof_sens_df["confidence_score"], oof_sens_df["error"])
print(f"\n=== Sensitivity (N={len(oof_sens_df)}, giữ confidence=0) ===")
print(f"Pooled r = {r_pooled_sens:.3f} (p={p_pooled_sens:.4f})")
for cls in CLASS_ORDER:
    sub = oof_sens_df[oof_sens_df["true_label"] == cls]
    r, p = pearsonr(sub["confidence_score"], sub["error"])
    print(f"  {cls}: r={r:.3f} p={p:.4f}")

In [ ]:
# =============================================================================
# CELL 23 — Xuất file dataset DepSeverity đã làm sạch (3.434 dòng), 3 cột
# =============================================================================
EXPORT_PATH = f"{BASE_DIR}/depseverity_clean_3434.csv"
export_df = depseverity_df[["text", "label", "confidence_score"]]

print(f"=== Dataset DepSeverity đã làm sạch: {len(export_df)} dòng (kỳ vọng 3.434) ===")
print(export_df["label"].value_counts().reindex(CLASS_ORDER))
if len(export_df) != 3434:
    print("CẢNH BÁO: số dòng không khớp 3.434 -- kiểm tra lại.")

export_df.to_csv(EXPORT_PATH, index=False)
print(f"\nĐã lưu: {EXPORT_PATH}")